In [0]:
# Configuración de credenciales para acceso al Blob Storage

spark.conf.set(
    "fs.azure.account.key.diplomaturastorage2026.blob.core.windows.net",
    "TU_KEY_AQUI"
)

In [0]:
# Definición de rutas Bronze, Silver y Rejected

ruta_bronze = "wasbs://bronze@diplomaturastorage2026.blob.core.windows.net/"
ruta_silver = "wasbs://silver@diplomaturastorage2026.blob.core.windows.net/"
ruta_rejected = "wasbs://rejected@diplomaturastorage2026.blob.core.windows.net/"
from pyspark.sql.functions import col, when, count

In [0]:
# Lectura de tablas Bronze

df_ventas_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "ventas")

df_productos_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "productos")

df_clientes_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "clientes")

df_tiendas_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "tiendas")

# Conversión de tipos en clientes

df_clientes_bronze = df_clientes_bronze \
    .withColumn("id_cliente", col("id_cliente").cast("int")) \
    .withColumn("edad", col("edad").cast("int"))

In [0]:
# Validación de cantidad de registros Bronze

print("Ventas Bronze:", df_ventas_bronze.count())
print("Productos Bronze:", df_productos_bronze.count())
print("Clientes Bronze:", df_clientes_bronze.count())
print("Tiendas Bronze:", df_tiendas_bronze.count())

Ventas Bronze: 2003
Productos Bronze: 10
Clientes Bronze: 100
Tiendas Bronze: 5


In [0]:
# Conversión de tipos de datos en ventas

df_ventas_typed = df_ventas_bronze \
    .withColumn("id_venta", col("id_venta").cast("int")) \
    .withColumn("id_producto", col("id_producto").cast("int")) \
    .withColumn("id_tienda", col("id_tienda").cast("int")) \
    .withColumn("id_cliente", col("id_cliente").cast("int")) \
    .withColumn("cantidad", col("cantidad").cast("int")) \
    .withColumn("precio_unitario", col("precio_unitario").cast("decimal(10,2)")) \
    .withColumn("fecha_venta", col("fecha_venta").cast("date"))

In [0]:
# Validación de estructura luego de conversión de tipos

df_ventas_typed.printSchema()

root
 |-- id_venta: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- id_producto: integer (nullable = true)
 |-- id_tienda: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(10,2) (nullable = true)



In [0]:
# Validación de duplicados en datasets

print("Duplicados ventas:")
display(
    df_ventas_typed
    .groupBy("id_venta")
    .count()
    .filter(col("count") > 1)
)

print("Duplicados productos:")
display(
    df_productos_bronze
    .groupBy("id_producto")
    .count()
    .filter(col("count") > 1)
)

print("Duplicados clientes:")
display(
    df_clientes_bronze
    .groupBy("id_cliente")
    .count()
    .filter(col("count") > 1)
)

print("Duplicados tiendas:")
display(
    df_tiendas_bronze
    .groupBy("id_tienda")
    .count()
    .filter(col("count") > 1)
)

Duplicados ventas:


id_venta,count
2002,2


Duplicados productos:


id_producto,count


Duplicados clientes:


id_cliente,count


Duplicados tiendas:


id_tienda,count


In [0]:
# Validación de valores nulos en ventas

df_ventas_typed.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_ventas_typed.columns
]).show()

+--------+-----------+-----------+---------+----------+--------+---------------+
|id_venta|fecha_venta|id_producto|id_tienda|id_cliente|cantidad|precio_unitario|
+--------+-----------+-----------+---------+----------+--------+---------------+
|       0|          0|          0|        0|         0|       0|              0|
+--------+-----------+-----------+---------+----------+--------+---------------+



In [0]:
# Validación de cantidades inválidas

df_cantidades_invalidas = df_ventas_typed \
    .filter(col("cantidad") <= 0)

display(df_cantidades_invalidas)

id_venta,fecha_venta,id_producto,id_tienda,id_cliente,cantidad,precio_unitario
2002,2026-05-02,101,2,1002,-3,2000.00
2002,2026-05-02,101,2,1002,-3,2000.00


In [0]:
# Validación de productos inexistentes

df_productos_invalidos = df_ventas_typed.join(
    df_productos_bronze,
    on="id_producto",
    how="left"
).filter(
    col("nombre_producto").isNull()
)

display(df_productos_invalidos)

id_producto,id_venta,fecha_venta,id_tienda,id_cliente,cantidad,precio_unitario,nombre_producto,categoria,marca
999,2001,2026-05-01,1,1001,2,1500.00,null,null,null


In [0]:
# Consolidación de registros inválidos

df_rejected = df_ventas_typed \
    .filter(
        (col("cantidad") <= 0)
    )

display(df_rejected)

id_venta,fecha_venta,id_producto,id_tienda,id_cliente,cantidad,precio_unitario
2002,2026-05-02,101,2,1002,-3,2000.00
2002,2026-05-02,101,2,1002,-3,2000.00


In [0]:
# Registros con productos inexistentes

df_rejected_productos = df_ventas_typed.join(
    df_productos_bronze,
    on="id_producto",
    how="left"
).filter(
    col("nombre_producto").isNull()
).select(df_ventas_typed["*"])

display(df_rejected_productos)

id_venta,fecha_venta,id_producto,id_tienda,id_cliente,cantidad,precio_unitario
2001,2026-05-01,999,1,1001,2,1500.00


In [0]:
# Identificación de registros duplicados

df_ids_duplicados = df_ventas_typed \
    .groupBy("id_venta") \
    .count() \
    .filter(col("count") > 1) \
    .select("id_venta")

df_rejected_duplicados = df_ventas_typed.join(
    df_ids_duplicados,
    on="id_venta",
    how="inner"
)

display(df_rejected_duplicados)

id_venta,fecha_venta,id_producto,id_tienda,id_cliente,cantidad,precio_unitario
2002,2026-05-02,101,2,1002,-3,2000.00
2002,2026-05-02,101,2,1002,-3,2000.00


In [0]:
# Consolidación final de registros rechazados

df_rejected_final = df_rejected \
    .union(df_rejected_productos) \
    .union(df_rejected_duplicados) \
    .dropDuplicates()

display(df_rejected_final)

id_venta,fecha_venta,id_producto,id_tienda,id_cliente,cantidad,precio_unitario
2002,2026-05-02,101,2,1002,-3,2000.00
2001,2026-05-01,999,1,1001,2,1500.00


In [0]:
# Construcción de dataset válido para Silver

df_ventas_silver = df_ventas_typed.join(
    df_rejected_final.select("id_venta"),
    on="id_venta",
    how="left_anti"
)

display(df_ventas_silver)

In [0]:
# Enriquecimiento dimensional de ventas Silver

df_ventas_silver_enriched = df_ventas_silver \
    .join(df_productos_bronze, on="id_producto", how="left") \
    .join(df_clientes_bronze, on="id_cliente", how="left") \
    .join(df_tiendas_bronze, on="id_tienda", how="left")

display(df_ventas_silver_enriched)

In [0]:
# Cálculo de importe total de venta

df_ventas_silver_enriched = df_ventas_silver_enriched \
    .withColumn(
        "importe_total",
        col("cantidad") * col("precio_unitario")
    )

display(df_ventas_silver_enriched)

In [0]:
from pyspark.sql.window import Window

# Segmento etario

df_ventas_silver_enriched = df_ventas_silver_enriched.withColumn(
    "segmento_edad",
    when(col("edad") < 25, "Joven")
    .when(col("edad") < 45, "Adulto")
    .otherwise("Senior")
)

# Tipo de ticket

df_ventas_silver_enriched = df_ventas_silver_enriched.withColumn(
    "tipo_ticket",
    when(col("importe_total") < 5000, "Bajo")
    .when(col("importe_total") < 15000, "Medio")
    .otherwise("Alto")
)

# Tipo de cliente según cantidad de compras

window_cliente = Window.partitionBy("id_cliente")

df_ventas_silver_enriched = df_ventas_silver_enriched.withColumn(
    "cantidad_compras_cliente",
    count("id_venta").over(window_cliente)
)

df_ventas_silver_enriched = df_ventas_silver_enriched.withColumn(
    "tipo_cliente",
    when(col("cantidad_compras_cliente") == 1, "Nuevo")
    .when(col("cantidad_compras_cliente") <= 5, "Frecuente")
    .otherwise("Premium")
)

display(df_ventas_silver_enriched)

In [0]:
# Métricas de calidad de datos

print("Cantidad registros válidos:", df_ventas_silver.count())
print("Cantidad registros rechazados:", df_rejected_final.count())
print("Cantidad duplicados:", df_rejected_duplicados.count())
print("Cantidad productos inválidos:", df_rejected_productos.count())
print("Cantidad cantidades inválidas:", df_rejected.count())

Cantidad registros válidos: 2000
Cantidad registros rechazados: 2
Cantidad duplicados: 2
Cantidad productos inválidos: 1
Cantidad cantidades inválidas: 2


In [0]:
# Escritura de registros rechazados en Delta Lake

df_rejected_final.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_rejected + "ventas_rejected")

In [0]:
# Escritura de tabla Silver enriquecida

# Eliminación física previa de tabla Silver

dbutils.fs.rm(ruta_silver + "ventas", recurse=True)

df_ventas_silver_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_silver + "ventas")

In [0]:
# Verificación de archivos almacenados en Silver y Rejected

print("Contenido Silver:")
display(dbutils.fs.ls(ruta_silver))

print("Contenido Rejected:")
display(dbutils.fs.ls(ruta_rejected))

Contenido Silver:


path,name,size,modificationTime
wasbs://silver@diplomaturastorage2026.blob.core.windows.net/ventas/,ventas/,0,1778525166000


Contenido Rejected:


path,name,size,modificationTime
wasbs://rejected@diplomaturastorage2026.blob.core.windows.net/ventas_rejected/,ventas_rejected/,0,1778525162000
